# Warm-up 2A · Traffic Camera Detection · Phát hiện phương tiện giao thông (Object Detection · Phát hiện đối tượng)

| | 🇬🇧 English | 🇻🇳 Tiếng Việt |
|---|---|---|
| **Task** | Detect every **motorbike, car, bus/truck, bicycle and pedestrian** in 320×320 frames from traffic cameras mounted above streets and intersections in Hanoi and Ho Chi Minh City: output a box, a class and a confidence score for each object. | Phát hiện mọi **xe máy, ô tô, xe buýt/xe tải, xe đạp và người đi bộ** trong các khung hình 320×320 từ camera giao thông đặt phía trên đường phố và ngã tư ở Hà Nội và TP.HCM: với mỗi đối tượng, đưa ra khung bao, lớp và điểm tin cậy. |
| **Data** | `train/` 2,000 images + `labels.csv` · `public_test/` 500 images · `private_test/` 500 images (no labels). Frames come from many cameras, at different times of day and in different weather. | `train/` 2.000 ảnh + `labels.csv` · `public_test/` 500 ảnh · `private_test/` 500 ảnh (không có nhãn). Các khung hình đến từ nhiều camera, ở nhiều thời điểm trong ngày và điều kiện thời tiết khác nhau. |
| **Metric** | **mAP@0.5** — mean over the 5 classes of the Average Precision at IoU ≥ 0.5, **higher is better**. Ranking uses the **private** score. | **mAP@0.5** — trung bình trên 5 lớp của Average Precision với ngưỡng IoU ≥ 0,5, **càng cao càng tốt**. Xếp hạng theo điểm **private**. |
| **Submission** | `public_submission.csv`, `private_submission.csv`: one row per predicted box, columns `image_id,class,score,x_min,y_min,x_max,y_max` (pixels). An image may have no rows; at most 100 boxes per image. | `public_submission.csv`, `private_submission.csv`: mỗi dòng là một khung dự đoán, gồm cột `image_id,class,score,x_min,y_min,x_max,y_max` (đơn vị pixel). Một ảnh có thể không có dòng nào; tối đa 100 khung mỗi ảnh. |
| **Rules** | PyTorch model; torchvision architectures and pretrained weights are allowed. No external data, no manual labelling of test images. | Mô hình PyTorch; được dùng kiến trúc và trọng số huấn luyện sẵn của torchvision. Không dùng dữ liệu ngoài, không tự gán nhãn ảnh test. |

$$\text{IoU}(\hat b,b)=\frac{|\hat b\cap b|}{|\hat b\cup b|},\qquad
\text{AP}_c=\int_0^1 \max_{r'\ge r} p_c(r')\,dr,\qquad
\text{mAP@0.5}=\frac{1}{5}\sum_{c}\text{AP}_c$$

A prediction is a true positive when it overlaps a not-yet-matched ground-truth box of the same class with IoU ≥ 0.5
(predictions are processed from the highest score down; duplicates count as false positives).
Một dự đoán là đúng (TP) khi nó trùng với một khung thật cùng lớp chưa được ghép với IoU ≥ 0,5
(xét dự đoán theo điểm từ cao xuống thấp; khung trùng lặp bị tính là sai — FP).

**Columns · Các cột** (`train/labels.csv`): `image_id, class` (`motorbike, car, bus_truck, bicycle, pedestrian`)`,
x_min, y_min, x_max, y_max` (pixels · pixel). Images: `<split>/images/<image_id>.jpg`; test ids are listed in
`public_test/public_test.csv` and `private_test/private_test.csv` · danh sách ảnh test nằm trong hai file này.

*Inspired by · Lấy cảm hứng từ:* [COCO Object Detection](https://cocodataset.org/#detection-eval) and the [NVIDIA AI City Challenge](https://www.aicitychallenge.org/) (2024–2025). All images are synthetic · Toàn bộ ảnh là dữ liệu tổng hợp.

🇬🇧 This baseline runs end-to-end and submits a result. Beating it is up to you.
🇻🇳 Baseline này chạy trọn vẹn và nộp được kết quả. Vượt qua nó là việc của bạn.

## 1. Setup and Imports · Cài đặt và thư viện

In [ ]:
import os
import re
import shutil
import unicodedata
import zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from PIL import Image
from torch.utils.data import Dataset
from torchvision.models import MobileNet_V3_Large_Weights
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_320_fpn
from torchvision.transforms.functional import to_tensor

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 7. Submit · Nộp bài

🇬🇧 Your team comes from your account on [seleccion.ftds.online](https://seleccion.ftds.online/), not from a name typed here. The cell below prints a short code once per Colab session; type it on the site and this notebook submits as you.

🇻🇳 Đội của bạn lấy từ tài khoản trên [seleccion.ftds.online](https://seleccion.ftds.online/). Ô bên dưới in ra một mã ngắn mỗi phiên Colab; nhập mã đó trên trang web.


In [ ]:
# ---- Data / Dữ liệu ----
# The dataset downloads itself from GitHub straight onto this Colab session's own
# disk. No Google Drive and no mounting: reading thousands of small files over a
# Drive mount is what made the old flow slow, and local disk is several times
# faster per epoch.
# Bộ dữ liệu tự tải từ GitHub vào ổ đĩa của phiên Colab này. Không cần Drive.
TASK    = "2A"
FOLDER  = "2A_traffic_detection"
RELEASE = "https://github.com/ftulabs/olympiad-datasets/releases/download/warmup-2"

# Already have the folder? Point this at the parent of 2A_traffic_detection/ and nothing is
# downloaded. / Đã có sẵn thư mục? Trỏ vào thư mục cha của 2A_traffic_detection/.
ROOT_OVERRIDE = ""

import os, subprocess, time

ROOT = ROOT_OVERRIDE or "/content/olympiad"
if not os.path.isdir(f"{ROOT}/{FOLDER}/dataset"):
    os.makedirs(ROOT, exist_ok=True)
    zip_path = f"{ROOT}/{FOLDER}.zip"
    if not os.path.exists(zip_path):
        print(f"Downloading {FOLDER}.zip / Đang tải ...", flush=True)
        t0 = time.time()
        subprocess.run(["curl", "-fsSL", "--retry", "3", "-o", zip_path,
                        f"{RELEASE}/{FOLDER}.zip"], check=True)
        print(f"  {os.path.getsize(zip_path)/1e6:.0f} MB in {time.time()-t0:.0f}s", flush=True)
    print("Extracting / Đang giải nén ...", flush=True)
    subprocess.run(["unzip", "-nq", zip_path, "-d", ROOT], check=True)

if not os.path.isdir(f"{ROOT}/{FOLDER}/dataset"):
    raise FileNotFoundError(
        f"{ROOT}/{FOLDER}/dataset is missing after the download. Re-run this cell. "
        f"/ Thiếu dữ liệu sau khi tải. Hãy chạy lại ô này."
    )
print(f"Data root / Thư mục dữ liệu: {ROOT}")

DATA_DIR = f"{ROOT}/2A_traffic_detection/dataset"
TRAIN_IMG_DIR = f"{DATA_DIR}/train/images"
TRAIN_LABELS = f"{DATA_DIR}/train/labels.csv"
PUBLIC_IMG_DIR = f"{DATA_DIR}/public_test/images"
PRIVATE_IMG_DIR = f"{DATA_DIR}/private_test/images"
PUBLIC_CSV = f"{DATA_DIR}/public_test/public_test.csv"
PRIVATE_CSV = f"{DATA_DIR}/private_test/private_test.csv"
RESULTS_DIR = f"{ROOT}/results"  # shared submission folder / thư mục nộp bài chung
SAVE_DIR = "./checkpoints"

# ---- Hyperparameters / Siêu tham số ----
IMG_SIZE = 320          # model input size / kích thước ảnh đầu vào mô hình
BATCH_SIZE = 8
EPOCHS = 10
LR = 0.005
SCORE_THRESHOLD = 0.5   # keep boxes above this score / giữ các khung có điểm cao hơn ngưỡng
NUM_WORKERS = 2
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

os.makedirs(SAVE_DIR, exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 3. Data Loading · Đọc dữ liệu

In [ ]:
CLASSES = ["motorbike", "car", "bus_truck", "bicycle", "pedestrian"]
CLASS_TO_ID = {c: i + 1 for i, c in enumerate(CLASSES)}  # 0 = background / 0 = nền
BOX_COLS = ["x_min", "y_min", "x_max", "y_max"]
SUB_COLUMNS = ["image_id", "class", "score", *BOX_COLS]

labels_df = pd.read_csv(TRAIN_LABELS)
train_ids = sorted(p.stem for p in Path(TRAIN_IMG_DIR).glob("*.jpg"))
public_ids = pd.read_csv(PUBLIC_CSV)["image_id"].tolist()
private_ids = pd.read_csv(PRIVATE_CSV)["image_id"].tolist()
print(f"Train images: {len(train_ids)} ({len(labels_df)} boxes) | Public test: {len(public_ids)} | Private test: {len(private_ids)}")


class TrafficDataset(Dataset):
    """Returns (image, target, image_id) in the format torchvision detectors expect.
    Trả về (ảnh, nhãn, image_id) theo đúng định dạng mô hình phát hiện của torchvision."""

    def __init__(self, img_dir, ids, labels=None):
        self.img_dir = img_dir
        self.ids = list(ids)
        self.groups = {k: g for k, g in labels.groupby("image_id")} if labels is not None else {}

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        image_id = self.ids[i]
        image = to_tensor(Image.open(f"{self.img_dir}/{image_id}.jpg").convert("RGB"))
        g = self.groups.get(image_id)
        if g is None:  # test image / ảnh test
            target = {"boxes": torch.zeros((0, 4)), "labels": torch.zeros((0,), dtype=torch.int64)}
        else:
            target = {"boxes": torch.tensor(g[BOX_COLS].values, dtype=torch.float32),
                      "labels": torch.tensor(g["class"].map(CLASS_TO_ID).values, dtype=torch.int64)}
        return image, target, image_id


def collate_fn(batch):
    """Images have different numbers of boxes, so keep lists. / Mỗi ảnh có số khung khác nhau nên giữ dạng list."""
    return tuple(zip(*batch))


# Validation = random 10% of the training images / Tập validation = 10% ảnh train chọn ngẫu nhiên
perm = np.random.default_rng(SEED).permutation(len(train_ids))
n_val = len(train_ids) // 10
val_ids = [train_ids[i] for i in perm[:n_val]]
tr_ids = [train_ids[i] for i in perm[n_val:]]
val_gt = labels_df[labels_df["image_id"].isin(val_ids)]

train_loader = DataLoader(TrafficDataset(TRAIN_IMG_DIR, tr_ids, labels_df), batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, collate_fn=collate_fn)
val_loader = DataLoader(TrafficDataset(TRAIN_IMG_DIR, val_ids, labels_df), batch_size=BATCH_SIZE,
                        num_workers=NUM_WORKERS, collate_fn=collate_fn)
print(f"Train: {len(tr_ids)} | Validation: {len(val_ids)}")
labels_df.head()

## 4. Model Architecture · Kiến trúc mô hình

In [ ]:
# Faster R-CNN with a MobileNetV3 backbone pretrained on ImageNet; the detection heads start from scratch.
# Faster R-CNN với backbone MobileNetV3 huấn luyện sẵn trên ImageNet; phần đầu phát hiện học từ đầu.
model = fasterrcnn_mobilenet_v3_large_320_fpn(
    weights=None,
    weights_backbone=MobileNet_V3_Large_Weights.IMAGENET1K_V1,
    num_classes=len(CLASSES) + 1,  # + background / + lớp nền
    min_size=IMG_SIZE,
    max_size=IMG_SIZE,
).to(DEVICE)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 5. Training · Huấn luyện

In [ ]:
@torch.no_grad()
def predict(model, loader, score_threshold=SCORE_THRESHOLD):
    """One row per predicted box. / Mỗi dòng là một khung dự đoán."""
    model.eval()
    rows = []
    for images, _, ids in loader:
        outputs = model([img.to(DEVICE) for img in images])
        for image_id, out in zip(ids, outputs):
            keep = out["scores"] >= score_threshold
            for box, label, score in zip(out["boxes"][keep].tolist(), out["labels"][keep].tolist(),
                                         out["scores"][keep].tolist()):
                rows.append([image_id, CLASSES[label - 1], round(score, 4), *[round(v, 1) for v in box]])
    return pd.DataFrame(rows, columns=SUB_COLUMNS)


def box_iou(box, boxes):
    ix = np.clip(np.minimum(box[2], boxes[:, 2]) - np.maximum(box[0], boxes[:, 0]), 0, None)
    iy = np.clip(np.minimum(box[3], boxes[:, 3]) - np.maximum(box[1], boxes[:, 1]), 0, None)
    inter = ix * iy
    union = (box[2] - box[0]) * (box[3] - box[1]) + (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1]) - inter
    return inter / np.maximum(union, 1e-9)


def mean_average_precision(pred, gt, iou_threshold=0.5):
    """mAP@0.5 over classes (VOC all-point AP), same rule as the leaderboard.
    mAP@0.5 trên các lớp (AP nội suy mọi điểm kiểu VOC), cùng quy tắc với bảng xếp hạng."""
    aps = []
    for c in CLASSES:
        g = gt[gt["class"] == c]
        if len(g) == 0:
            continue
        p = pred[pred["class"] == c].sort_values("score", ascending=False, kind="mergesort")
        gt_boxes = {k: v[BOX_COLS].values.astype(float) for k, v in g.groupby("image_id")}
        used = {k: np.zeros(len(v), bool) for k, v in gt_boxes.items()}
        tp = np.zeros(len(p), bool)
        for i, (image_id, *box) in enumerate(p[["image_id", *BOX_COLS]].itertuples(index=False)):
            if image_id in gt_boxes:
                ious = np.where(used[image_id], -1.0, box_iou(np.array(box, float), gt_boxes[image_id]))
                j = int(np.argmax(ious))
                if ious[j] >= iou_threshold:
                    used[image_id][j] = tp[i] = True
        ctp, cfp = np.cumsum(tp), np.cumsum(~tp)
        recall = np.concatenate([[0.0], ctp / len(g), [1.0]])
        precision = np.concatenate([[0.0], ctp / np.maximum(ctp + cfp, 1), [0.0]])
        precision = np.maximum.accumulate(precision[::-1])[::-1]
        k = np.nonzero(recall[1:] != recall[:-1])[0]
        aps.append(float(np.sum((recall[k + 1] - recall[k]) * precision[k + 1])))
    return float(np.mean(aps))


params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=LR, momentum=0.9, weight_decay=1e-4)

best_map = -1.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for images, targets, _ in train_loader:
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        loss = sum(model(images, targets).values())  # classification + box-regression losses / các hàm mất mát
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    val_map = mean_average_precision(predict(model, val_loader), val_gt)
    msg = f"Epoch {epoch:02d} | Loss={total_loss / len(train_loader):.4f} | Val mAP@0.5={val_map:.4f}"
    if val_map > best_map:
        best_map = val_map
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best_model.pt"))
        msg += "  ✓ best model saved"
    print(msg)

print(f"\nBest validation mAP@0.5: {best_map:.4f}")

## 6. Predict Public & Private Test · Dự đoán tập Public & Private
🇬🇧 Creates `public_submission.csv` / `private_submission.csv` (+ `.zip`), like the official competition.
🇻🇳 Tạo `public_submission.csv` / `private_submission.csv` (+ `.zip`), giống định dạng cuộc thi chính thức.

In [ ]:
def export(pred_df, csv_path):
    """Save predicted boxes as CSV and zip it. / Lưu các khung dự đoán ra CSV và nén zip."""
    pred_df.to_csv(csv_path, index=False)
    zip_path = csv_path.replace(".csv", ".zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(csv_path, arcname=os.path.basename(csv_path))
    print(f"Saved {csv_path} ({len(pred_df)} boxes) and {zip_path}")


model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "best_model.pt"), map_location=DEVICE, weights_only=True))
public_loader = DataLoader(TrafficDataset(PUBLIC_IMG_DIR, public_ids), batch_size=BATCH_SIZE,
                           num_workers=NUM_WORKERS, collate_fn=collate_fn)
private_loader = DataLoader(TrafficDataset(PRIVATE_IMG_DIR, private_ids), batch_size=BATCH_SIZE,
                            num_workers=NUM_WORKERS, collate_fn=collate_fn)
export(predict(model, public_loader), "public_submission.csv")
export(predict(model, private_loader), "private_submission.csv")
pd.read_csv("public_submission.csv").head()

## 7. Submit · Nộp bài
🇬🇧 Checks the format, then copies both files to the shared results folder
`results/<task>/<team>/<timestamp>/`. You may submit several times; each run creates a new entry.
🇻🇳 Kiểm tra định dạng, sau đó sao chép hai file vào thư mục kết quả chung
`results/<task>/<team>/<thời điểm>/`. Có thể nộp nhiều lần; mỗi lần chạy tạo một lượt nộp mới.

In [ ]:
def check_submission(csv_path, test_ids):
    """Validate the file format before submitting. / Kiểm tra định dạng file trước khi nộp."""
    sub = pd.read_csv(csv_path)
    assert list(sub.columns) == SUB_COLUMNS, f"{csv_path}: columns must be {SUB_COLUMNS}"
    assert set(sub["image_id"]) <= set(test_ids), f"{csv_path}: unknown image_id / có image_id không thuộc tập test"
    assert sub["class"].isin(CLASSES).all(), f"{csv_path}: unknown class / có lớp không hợp lệ"
    assert sub[["score", *BOX_COLS]].notna().all().all(), f"{csv_path}: contains missing values / có giá trị bị thiếu"
    assert (sub["image_id"].value_counts() <= 100).all(), f"{csv_path}: > 100 boxes per image / quá 100 khung mỗi ảnh"
    print(f"{csv_path}: {len(sub)} boxes on {sub['image_id'].nunique()}/{len(test_ids)} images — OK")


check_submission("public_submission.csv", public_ids)
check_submission("private_submission.csv", private_ids)


In [ ]:
# ===================================================================
#  Olympiad AI - submit.  Paste at the END of your notebook, then run.
#  No key to copy: the first run prints a code you enter on the site.
# ===================================================================
BASE    = "https://seleccion.ftds.online"
PROBLEM = "2A"          # which problem this notebook is for
CONTEST = ""            # blank = the contest you are entered in
PUBLIC  = "public_submission.csv"
PRIVATE = "private_submission.csv"
# -------------------------------------------------------------------
import os, time, requests

token = globals().get("_OLY_TOKEN")
if not token:
    start = requests.post(f"{BASE}/api/pair/start", timeout=30).json()
    print(f"\n  1. open   {start['enter_at']}")
    print(f"  2. enter  {start['code']}\n  waiting...", flush=True)
    for _ in range(int(start["expires_in"]) // 3):
        time.sleep(3)
        got = requests.get(f"{BASE}/api/pair/poll",
                           params={"code": start["code"]}, timeout=30).json()
        if got.get("ready"):
            token = globals()["_OLY_TOKEN"] = got["token"]
            print("  paired ✓ (this notebook stays paired for the session)")
            break
    else:
        raise SystemExit("pairing timed out - run this cell again")

missing = [f for f in (PUBLIC, PRIVATE) if not os.path.exists(f)]
if PUBLIC in missing:
    raise SystemExit(f"{PUBLIC} not found - run the cell that writes it first")

files = {"public": open(PUBLIC, "rb")}
if PRIVATE not in missing:
    files["private"] = open(PRIVATE, "rb")
try:
    r = requests.post(f"{BASE}/api/submit", headers={"X-Oly-Token": token},
                      data={"problem": PROBLEM, "contest": CONTEST},
                      files=files, timeout=600)
finally:
    for f in files.values():
        f.close()

try:
    d = r.json()
except ValueError:
    raise SystemExit(f"server {r.status_code}: {r.text[:300]}")

if r.status_code == 429:
    print(f"⏳ cooldown - wait {d.get('retry_after')}s and run again")
elif r.status_code == 401:
    globals().pop("_OLY_TOKEN", None)
    print(f"❌ {d.get('detail')}  (run this cell again to re-pair)")
elif r.status_code != 200:
    print(f"❌ {r.status_code}: {d.get('detail') or d.get('message')}")
elif d["public"]["status"] != "ok":
    # the grader's own message about your file - it says what to fix
    print(f"❌ {d['public']['status'].upper()}: {d['public']['message']}")
else:
    p, c = d["public"], (d.get("counting") or {})
    print(f"✅ {d['problem']} · {d['title']}")
    print(f"   {d['metric']} = {p['score']:.4f}   {p['tier'] or '(below the lowest tier)'}")
    print(f"   entry #{d['entries']}   ·   rank: {d['rank']}")
    if c.get("score") is not None and abs(c["score"] - p["score"]) > 1e-12:
        print(f"   counting entry: {c['score']:.4f} {c['tier']}")
    print(f"   {BASE}/")
